In [0]:
from pyspark.sql import functions as F

sessions = spark.table("retailnew.silver.clickstream_sessions")

fact = (
    sessions.groupBy("session_id", "user_id")
    .agg(
        F.min("event_time").alias("session_start"),
        F.max("event_time").alias("session_end"),
        F.count("*").alias("event_count")
    )
    .withColumn(
        "session_duration_sec",
        F.unix_timestamp("session_end") - F.unix_timestamp("session_start")
    )
)

(
    fact.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("retailnew.gold.fact_sessions", overwriteSchema=True)
)

